# Persona Menu Ingestion to Azure AI Search

## 1: Notebook Introduction
This notebook demonstrates how to ingest one persona's menu (`personas/<PERSONA_ID>/`) into
Azure AI Search:
1. Configure Azure OpenAI and Azure AI Search services for the target persona (set `PERSONA_ID`
   below -- everything else in this notebook reads the persona's own `persona.json`/raw export,
   nothing is hardcoded to a specific brand).
2. Prepare the persona's menu JSON data for ingestion into Azure AI Search, and write the
   transformed data into `personas/<PERSONA_ID>/menu/menuItems.json`.
3. Upload the prepared data to Azure AI Search for hybrid semantic search capabilities.

This notebook defaults `PERSONA_ID` to `sonic`, since Sonic is the only persona with a raw
production export checked into this repo today -- change the one `PERSONA_ID` variable in the
imports cell to run it for another persona's export.

## 2: Install Required Packages

### Description
This cell installs all the necessary packages required for the notebook.
It ensures that all dependencies are met before proceeding with the rest of the notebook.


In [ ]:
%pip install azure-core azure-search-documents python-dotenv langchain-openai langchain-community openai==1.54.3 "httpx>=0.23.0,<0.28.0" pydantic tenacity

## 3: Imports and Environment Setup

### Description
This cell imports necessary libraries and loads environment variables using `dotenv`.
Ensure your `.env` file is properly set up with the required Azure API keys and endpoints.


In [ ]:
# Import required libraries
import json
import os
import re

from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    VectorSearch,
    VectorSearchProfile,
)
from dotenv import load_dotenv
from openai import AzureOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

# Which persona pack (personas/<PERSONA_ID>/) this run ingests -- change this one variable to
# repoint every cell below at a different persona; nothing else in this notebook names a brand.
PERSONA_ID = 'sonic'

# Load environment variables from the repo backend .env (fallback to current dir)
env_path = os.path.join('..', 'app', 'backend', '.env')
load_dotenv(dotenv_path=env_path)
load_dotenv()  # allows overriding with local .env if present
print(f'Loaded env file at {env_path}: {os.path.exists(env_path)}')
print(f'PERSONA_ID: {PERSONA_ID}')

## 4: Azure OpenAI and Azure AI Search Configuration

### Description
This cell sets up the Azure OpenAI and AI Search configurations.
Ensure that the endpoints and deployment names in the `.env` file match your Azure resource
setup. Authentication uses `DefaultAzureCredential` only -- no Azure OpenAI or Search key is
ever read, issued, or stored.

**Note:** The target index name is read from `personas/<PERSONA_ID>/persona.json`
(`search.indexName`), never hardcoded here.

In [ ]:
# Quick config sanity check (prints masked values -- no keys are used, so nothing sensitive
# ends up in this output)
def _mask(val: str):
    if not val:
        return '<missing>'
    return f'len={len(val)}, suffix={val[-4:]}'

persona_dir = os.path.join('..', 'personas', PERSONA_ID)
persona_manifest_path = os.path.join(persona_dir, 'persona.json')
if not os.path.exists(persona_manifest_path):
    raise FileNotFoundError(
        f'No persona.json found for PERSONA_ID={PERSONA_ID!r} at {persona_manifest_path}'
    )

with open(persona_manifest_path, encoding='utf-8') as f:
    persona_manifest = json.load(f)

index_name = persona_manifest['search']['indexName']

vars_to_check = {
    'AZURE_OPENAI_EASTUS_ENDPOINT': os.getenv('AZURE_OPENAI_EASTUS_ENDPOINT'),
    'AZURE_OPENAI_API_VERSION': os.getenv('AZURE_OPENAI_API_VERSION'),
    'AZURE_OPENAI_EMBEDDING_DEPLOYMENT': os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT'),
    'AZURE_SEARCH_ENDPOINT': os.getenv('AZURE_SEARCH_ENDPOINT'),
}

for name, value in vars_to_check.items():
    print(f'{name}: {_mask(value)}')

missing = [k for k, v in vars_to_check.items() if not v]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

print('All required environment variables are present.')
print(f'PERSONA_ID={PERSONA_ID!r} -> index_name={index_name!r} (read from {persona_manifest_path})')

In [ ]:
# Azure OpenAI setup
aoai_eastus_endpoint = os.getenv('AZURE_OPENAI_EASTUS_ENDPOINT')
aoai_openai_api_version = os.getenv('AZURE_OPENAI_API_VERSION')
aoai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')

# DefaultAzureCredential only -- no Azure OpenAI or Search key is ever read, issued, or stored.
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')

# Initialize the Azure OpenAI client
aoai_client = AzureOpenAI(
    azure_endpoint=aoai_eastus_endpoint,
    api_version=aoai_openai_api_version,
    azure_ad_token_provider=token_provider,
)

# index_name was read from personas/<PERSONA_ID>/persona.json above -- never hardcoded here.
search_service_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
search_client = SearchClient(endpoint=search_service_endpoint, index_name=index_name, credential=credential)
search_index_client = SearchIndexClient(endpoint=search_service_endpoint, credential=credential)

print('Azure OpenAI and Azure Search clients initialized successfully.')
print(f'Target index: {index_name}')

## 5: Define and Create/Update Index Schema with Semantic Configuration

### Description
This cell defines the schema for the target persona's Azure AI Search index, including fields
for semantic search and vector search capabilities. It deletes any existing index with the same
name and creates a fresh one.

### Note on enabling semantic ranker

Semantic search is a service-level capability. If you see errors like "semantic search is not enabled," ensure the service has semantic ranker turned on (Standard tier or higher). You can enable it via CLI if the portal UI does not surface a toggle:

`az search service update --name <service> --resource-group <rg> --semantic-search standard`

Once enabled, the existing index schema here (with `semantic_search` and `menuSemanticConfig`) works without further changes.


In [ ]:
# Define and Create/Update Index Schema with Semantic Configuration
if not index_name:
    raise RuntimeError('index_name is not set.')

index_schema = SearchIndex(
    name=index_name,
    fields=[
        SimpleField(name='id', type=SearchFieldDataType.String, key=True, sortable=True, filterable=True),
        SearchField(name='category', type=SearchFieldDataType.String, sortable=True, filterable=True, facetable=True),
        SearchField(name='name', type=SearchFieldDataType.String, sortable=True, filterable=True, facetable=True),
        SearchField(name='description', type=SearchFieldDataType.String),
        SearchField(name='longDescription', type=SearchFieldDataType.String),
        SearchField(name='origin', type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchField(name='caffeineContent', type=SearchFieldDataType.String, filterable=True),
        SearchField(name='brewingMethod', type=SearchFieldDataType.String, filterable=True),
        SearchField(name='popularity', type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchField(name='image', type=SearchFieldDataType.String, filterable=False, facetable=False),
        SearchField(name='sizes', type=SearchFieldDataType.String, filterable=False, facetable=False),
        SearchField(
            name='embedding',
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=3072,
            vector_search_profile_name='menuHnswProfile'
        )
    ],
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name='menuHnsw',
                kind='hnsw',
                parameters={'m': 10, 'efConstruction': 200}
            )
        ],
        profiles=[
            VectorSearchProfile(
                name='menuHnswProfile',
                algorithm_configuration_name='menuHnsw',
                vectorizer_name='menuVectorizer'
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name='menuVectorizer',
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=aoai_eastus_endpoint,
                    deployment_name=aoai_embedding_deployment,
                    model_name=aoai_embedding_deployment
                )
            )
        ]
    ),
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name='menuSemanticConfig',
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name='name'),
                    content_fields=[
                        SemanticField(field_name='description'),
                        SemanticField(field_name='longDescription'),
                        SemanticField(field_name='category')
                    ]
                )
            )
        ]
    ),
)

# Delete the existing index if it exists
try:
    search_index_client.delete_index(index_name)
    print(f'Deleted existing index: {index_name}')
except Exception as e:
    print(f'Index {index_name} does not exist or could not be deleted: {e}')

# Create the index schema
try:
    search_index_client.create_or_update_index(index=index_schema)
    print(f'Created index: {index_name}')
except Exception as e:
    raise RuntimeError(f'Failed to create/update index {index_name}: {e}')

# Verify index exists
existing = [i.name for i in search_index_client.list_indexes()]
print(f'Indexes present: {existing}')
if index_name not in existing:
    raise RuntimeError(f'Index {index_name} was not found after creation.')

## 6: Load and Process Production Menu Data

### Description
This cell reads the raw production `<PERSONA_ID>-menu-items.json` export, which uses a nested
structure with `menus`, `products`, `categories`, and `productGroups`. It recursively traverses
categories to find leaf products, resolves size variants from productGroups, and builds the
structured menu items list -- then writes that list into
`personas/<PERSONA_ID>/menu/menuItems.json` so the running app and `setup_search_index.py` both
read the same, already-schema-valid pack data.

In [ ]:
# Load the raw production menu export for this persona (nested menus/products/categories/
# productGroups format -- the legacy shape some brand exports arrive in). Only Sonic has one of
# these checked into the repo today; point this at your persona's own raw export before running
# with a different PERSONA_ID.
menu_data_candidates = [
    os.path.join('..', 'app', 'frontend', 'src', 'data', f'{PERSONA_ID}-menu-items.json'),
]

menu_data_path = None
for candidate in menu_data_candidates:
    if os.path.exists(candidate):
        menu_data_path = candidate
        break

if not menu_data_path:
    raise FileNotFoundError(
        f'Could not find a raw menu export for PERSONA_ID={PERSONA_ID!r}. Tried: {menu_data_candidates}'
    )

print(f'Using raw menu export at: {menu_data_path}')

with open(menu_data_path, encoding='utf-8') as file:
    raw_data = json.load(file)

# Access first menu dynamically
menu = list(raw_data['menus'].values())[0]
products = menu['products']
categories = menu['categories']
product_groups = menu['productGroups']

print(f'Menu: {menu.get("displayName", "unknown")}')
print(f'Total products: {len(products)}, Categories: {len(categories)}, ProductGroups: {len(product_groups)}')


def collect_products_from_category(cat_id, categories, leaf_name=None):
    """Recursively traverse categories, collecting (product_id, leaf_category_name) tuples."""
    cat = categories.get(cat_id)
    if not cat:
        return []
    current_name = leaf_name or cat.get('displayName', cat_id)
    result = []
    for ref in cat.get('childRefs', {}).keys():
        if ref.startswith('categories.'):
            sub_cat_id = ref[len('categories.'):]
            # Use the subcategory's own name as the leaf name
            sub_cat = categories.get(sub_cat_id)
            sub_name = sub_cat.get('displayName', sub_cat_id) if sub_cat else sub_cat_id
            result.extend(collect_products_from_category(sub_cat_id, categories, sub_name))
        elif ref.startswith('products.'):
            prod_id = ref[len('products.'):]
            result.append((prod_id, current_name))
    return result


# Collect all products referenced from top-level categories
product_category_map = {}  # product_id -> leaf category name

# Find top-level categories (not referenced as child of another category)
child_cat_ids = set()
for cat in categories.values():
    for ref in cat.get('childRefs', {}).keys():
        if ref.startswith('categories.'):
            child_cat_ids.add(ref[len('categories.'):])

top_level_cats = [cid for cid in categories.keys() if cid not in child_cat_ids]
print(f'Top-level categories: {len(top_level_cats)}')

for cat_id in top_level_cats:
    for prod_id, leaf_cat_name in collect_products_from_category(cat_id, categories):
        if prod_id not in product_category_map:
            product_category_map[prod_id] = leaf_cat_name

print(f'Products found via category traversal: {len(product_category_map)}')


def normalize_size_name(child_display_name, parent_display_name):
    """Extract clean size label from verbose product-specific display name."""
    prefixes = [
        ("Mini ", "Mini"),
        ("Sm ", "Small"),
        ("Med ", "Medium"),
        ("Lg ", "Large"),
        ("RT 44", "RT 44"),
        ("Rt. 44", "RT 44"),
    ]
    for prefix, label in prefixes:
        if child_display_name.startswith(prefix):
            return label
    # Fallback: strip parent name from child name
    stripped = child_display_name.replace(parent_display_name, '').strip()
    # Remove ® and other trademark symbols
    stripped = stripped.replace('\u00ae', '').strip()
    if stripped:
        return stripped
    return "Standard"


def get_size_variants(product, products, product_groups):
    """Resolve size variants for a product from relatedProducts.alternatives."""
    alternatives = product.get('relatedProducts', {}).get('alternatives', {})
    sizes = []
    for group_ref in alternatives.keys():
        if group_ref.startswith('productGroups.'):
            group_id = group_ref[len('productGroups.'):]
            group = product_groups.get(group_id)
            if not group:
                continue
            for child_ref in group.get('childRefs', {}).keys():
                if child_ref.startswith('products.'):
                    child_id = child_ref[len('products.'):]
                    child_product = products.get(child_id)
                    if child_product:
                        sizes.append({
                            'size': normalize_size_name(child_product.get('displayName', child_id), product.get('displayName', '')),
                            'price': child_product.get('price', 0.0)
                        })
    return sizes


# Build structured menu items
structured_menu_items = []

for prod_id, cat_name in product_category_map.items():
    product = products.get(prod_id)
    if not product:
        continue
    # Filter: only non-recipe products
    if product.get('isRecipe', False):
        continue

    size_variants = get_size_variants(product, products, product_groups)
    if not size_variants:
        # No size alternatives -- use product's own price
        price = product.get('price', 0.0) or 0.0
        size_variants = [{'size': 'Standard', 'price': price}]

    structured_menu_items.append({
        'productId': prod_id,
        'name': product.get('displayName', prod_id),
        'description': product.get('description', ''),
        'imageUrl': product.get('imageUrl', ''),
        'category': cat_name,
        'sizes': size_variants,
    })

print(f'\nTotal menu items for ingestion: {len(structured_menu_items)}')

# Show category distribution
from collections import Counter  # noqa: E402, I001

cat_counts = Counter(item['category'] for item in structured_menu_items)
print(f'\nCategory distribution ({len(cat_counts)} categories):')
for cat, count in cat_counts.most_common():
    print(f'  {cat}: {count}')

# Preview first 3 items
print('\n--- Sample items ---')
for item in structured_menu_items[:3]:
    print(json.dumps(item, indent=2))

# Write the transformed menu into this persona's own pack (issue #84: "the notebooks write to
# personas/<id>/menu/") so setup_search_index.py's ingestion (and the running app) both read the
# same, already-schema-valid data -- no separate hand-authored copy to keep in sync. Only
# 'name', 'sizes', and 'description' are required by personas/menu.schema.json; every other
# field here is optional context this raw export happens to provide.
menu_by_category: dict[str, list[dict]] = {}
for item in structured_menu_items:
    menu_by_category.setdefault(item['category'], []).append({
        'name': item['name'],
        'sizes': item['sizes'],
        'description': item['description'],
        'longDescription': item['description'],
        'origin': f'{PERSONA_ID.title()} Menu',
        'popularity': '',
        'image': item['imageUrl'],
        'aliases': [],
        'isExtra': False,
    })

pack_menu_dir = os.path.join(persona_dir, 'menu')
os.makedirs(pack_menu_dir, exist_ok=True)
pack_menu_path = os.path.join(pack_menu_dir, 'menuItems.json')
pack_menu_payload = {
    'menuItems': [
        {'category': category, 'items': items}
        for category, items in menu_by_category.items()
    ]
}
with open(pack_menu_path, 'w', encoding='utf-8') as f:
    json.dump(pack_menu_payload, f, indent=2)
    f.write('\n')

print(
    f'Wrote {sum(len(v) for v in menu_by_category.values())} item(s) across '
    f'{len(menu_by_category)} categories to {pack_menu_path}'
)

## 7: Data Preparation and Embedding Generation

### Description
This cell defines functions to sanitize document keys, generate embeddings using Azure OpenAI,
and prepare the persona's menu data for ingestion into Azure AI Search. It includes retry logic
for embedding generation and assigns embeddings to each document.

In [ ]:
def sanitize_key(key):
    """Sanitize the document key to contain only valid characters."""
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', key)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=5, max=60))
def generate_embeddings(texts):
    """Generate embeddings using Azure OpenAI with retry logic for a batch of texts."""
    response = aoai_client.embeddings.create(input=texts, model=aoai_embedding_deployment)
    return [res.embedding for res in response.data]


def generate_embeddings_batched(texts, batch_size=100):
    """Generate embeddings in batches to respect API limits."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        print(f'  Generating embeddings for batch {i // batch_size + 1} ({len(batch)} items)...')
        embeddings = generate_embeddings(batch)
        all_embeddings.extend(embeddings)
    return all_embeddings


def prepare_data_for_azure_search(menu_items):
    """Transform this persona's production menu data for Azure AI Search ingestion."""
    azure_search_documents = []
    texts_for_embedding = []

    for item in menu_items:
        doc_id = sanitize_key(item['productId'])
        name = item['name']
        category = item['category']
        description = item.get('description', '')
        image = item.get('imageUrl', '')

        combined_text = f"{name} {category} {description}"
        texts_for_embedding.append(combined_text)

        azure_search_documents.append({
            'id': doc_id,
            'name': name,
            'category': category,
            'description': description,
            'longDescription': description,
            'origin': '',
            'caffeineContent': '',
            'brewingMethod': '',
            'popularity': '',
            'image': image,
            'sizes': json.dumps(item['sizes']),
        })

    # Generate embeddings in batches
    print(f'Generating embeddings for {len(texts_for_embedding)} items...')
    embeddings = generate_embeddings_batched(texts_for_embedding, batch_size=100)

    for i, embedding in enumerate(embeddings):
        azure_search_documents[i]['embedding'] = embedding

    return azure_search_documents


documents_for_index = prepare_data_for_azure_search(structured_menu_items)

print(f'\nPrepared {len(documents_for_index)} documents for indexing.')
print('\n--- Sample document ---')
if documents_for_index:
    doc = documents_for_index[0]
    print(f"ID: {doc['id']}")
    print(f"Name: {doc['name']}")
    print(f"Category: {doc['category']}")
    print(f"Description: {doc['description'][:100]}")
    print(f"Image: {doc['image'][:80]}...")
    print(f"Sizes: {doc['sizes']}")
    print(f"Embedding dims: {len(doc['embedding'])}")

## 8: Upload to Azure AI Search

### Description
This cell uploads the prepared menu data to Azure AI Search.
Ensure the Azure AI Search index is properly configured before running this step.

In [ ]:
def upload_documents_to_search(documents):
    batch_size = 15
    total_batches = (len(documents) + batch_size - 1) // batch_size
    successful_uploads = 0

    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        try:
            _ = search_client.upload_documents(documents=batch)
            successful_uploads += len(batch)
            print(f'Uploaded batch {i // batch_size + 1}/{total_batches} successfully. Batch size: {len(batch)}')
        except HttpResponseError as e:
            print(f'Error uploading batch {i // batch_size + 1}/{total_batches}: {e}')
            continue

    print(f'\n{PERSONA_ID} menu items uploaded successfully. Total: {successful_uploads}/{len(documents)}')

print(f'Uploading {len(documents_for_index)} documents to index: {index_name}')
upload_documents_to_search(documents_for_index)

## 9: Testing Search Capabilities

### Summary
Test the target persona's menu search index with various query types:
- Simple text-based search for "burger"
- Semantic search for "frozen drinks with fruit flavors"
- Vector search for "crispy chicken sandwich with pickles"
- Hybrid search combining text and vector for "cold refreshing slush"
- Filter-based search for items in the "Shakes" category

(These sample queries are Sonic-flavored since `sonic` is this notebook's default `PERSONA_ID` --
swap them for queries relevant to your persona's own menu.)

In [ ]:
# Test Azure AI Search with various query types
%pip install --upgrade azure-search-documents
from azure.search.documents.models import VectorizedQuery


def perform_simple_search(query_text, filter_condition=None, top=5):
    """Perform a simple text-based search against the target persona's menu index"""
    results = search_client.search(
        search_text=query_text,
        filter=filter_condition,
        select=['id', 'name', 'description', 'category', 'sizes', 'image'],
        top=top,
        include_total_count=True
    )

    print(f"\n=== Simple Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_semantic_search(query_text, top=5):
    """Perform a semantic search using the configured semantic settings"""
    try:
        results = search_client.search(
            search_text=query_text,
            select=['id', 'name', 'description', 'category', 'sizes', 'image'],
            top=top,
            query_type='semantic',
            semantic_configuration_name='menuSemanticConfig',
            include_total_count=True
        )
    except HttpResponseError as e:
        print('Semantic search is not enabled for this search service. Enable it in the Azure portal or use simple search instead.')
        print(f'Details: {e}')
        return None

    print(f"\n=== Semantic Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_vector_search(query_text, top=5):
    """Generate embedding for the query and perform vector search"""
    client = AzureOpenAI(
        azure_ad_token_provider=token_provider,
        api_version='2023-05-15',
        azure_endpoint=aoai_eastus_endpoint
    )

    response = client.embeddings.create(input=query_text, model=aoai_embedding_deployment)
    query_vector = response.data[0].embedding

    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=top, fields='embedding')
    results = search_client.search(
        search_text=None,
        select=['id', 'name', 'description', 'category', 'sizes', 'image'],
        top=top,
        vector_queries=[vector_query],
        include_total_count=True
    )

    print(f"\n=== Vector Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_hybrid_search(query_text, top=5):
    """Perform a hybrid search combining vector search with text search"""
    client = AzureOpenAI(
        azure_ad_token_provider=token_provider,
        api_version='2023-05-15',
        azure_endpoint=aoai_eastus_endpoint
    )

    response = client.embeddings.create(input=query_text, model=aoai_embedding_deployment)
    query_vector = response.data[0].embedding

    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=top, fields='embedding')
    results = search_client.search(
        search_text=query_text,
        select=['id', 'name', 'description', 'category', 'sizes', 'image'],
        top=top,
        vector_queries=[vector_query],
        include_total_count=True
    )

    print(f"\n=== Hybrid Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


# Test different search types against this persona's menu (sample queries assume the default
# PERSONA_ID='sonic' -- adjust them if you've pointed this notebook at another persona)
print(f'\n========= TESTING {PERSONA_ID.upper()} SEARCH CAPABILITIES =========')

# Test 1: Simple text search
simple_results = perform_simple_search('burger')

# Test 2: Semantic search for natural language queries
semantic_results = perform_semantic_search('frozen drinks with fruit flavors')

# Test 3: Vector search for semantic similarity
vector_results = perform_vector_search('crispy chicken sandwich with pickles')

# Test 4: Hybrid search combining text and vector capabilities
hybrid_results = perform_hybrid_search('cold refreshing slush')

# Test 5: Search with category filter
filtered_results = perform_simple_search(
    'shake',
    filter_condition="category eq 'Classic Shakes'"
)

# Test 6: Banana Shake -- verify sizes field has valid JSON with prices
print('\n========= BANANA SHAKE SIZE VERIFICATION =========')
shake_results = search_client.search(
    search_text='Banana Shake',
    select=['id', 'name', 'category', 'sizes', 'image'],
    top=3,
    include_total_count=True
)
for result in shake_results:
    print(f"\nName: {result['name']}")
    print(f"Category: {result['category']}")
    sizes_data = json.loads(result['sizes'])
    print(f"Sizes ({len(sizes_data)} variants):")
    for s in sizes_data:
        print(f"  {s['size']}: ${s['price']:.2f}")
    print(f"Image: {result.get('image', 'N/A')[:80]}")
    print('-' * 50)

## 10: Final Summary

### Summary
- Installed required packages.
- Configured Azure OpenAI and Azure AI Search services using `DefaultAzureCredential` (no keys).
- Read the target index name from `personas/<PERSONA_ID>/persona.json` and defined/created that
  index's schema with semantic configuration (including the `image` field).
- Loaded and processed the persona's raw production menu export
  (`<PERSONA_ID>-menu-items.json`) and wrote the transformed data to
  `personas/<PERSONA_ID>/menu/menuItems.json`.
- Recursively traversed categories to extract non-recipe products with leaf-level category
  assignments.
- Resolved size variants (Mini/Small/Medium/Large/RT 44/Standard) from product group
  relationships.
- Generated embeddings using Azure OpenAI's `text-embedding-3-large` model.
- Uploaded all documents to Azure AI Search with retry logic for reliability.
- Tested multiple search modalities: simple text, semantic, vector, and hybrid search.
- Verified the `sizes` field returns valid JSON with correct pricing data.

This notebook's default `PERSONA_ID` is `sonic` -- re-run it with a different value (and that
persona's own raw export) to ingest another persona's menu.